In [1]:
# ── Step 1: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# ── Step 2: Configure paths ──────────────────────────────────────────────────
import pathlib

DRIVE_FOLDER      = pathlib.Path('/content/drive/MyDrive/ShapResearch/Woodelf_package/hybrid_woodelf_experiments/woodelf_sparse_variants/')
DRIVE_CACHE       = DRIVE_FOLDER / 'method_cache'
EXPERIMENT_NAME   = 'woodelf_sparse_variants'
EXPERIMENT_MODULE = 'benchmarks.woodelf_sparse_variants'

DRIVE_FOLDER.mkdir(parents=True, exist_ok=True)
DRIVE_CACHE.mkdir(parents=True, exist_ok=True)
print(f'Drive folder: {DRIVE_FOLDER}')

In [ ]:
# ── Step 3: Clone repositories ───────────────────────────────────────────────
TREEBRANCHMARKS_URL = 'https://github.com/ron-wettenstein/TreeBranchMarks.git'

# woodelf: clone the feature branch directly
!git clone {TREEBRANCHMARKS_URL} /content/treebranchmarks
!git clone -b feature/hybrid_background_shap \
    https://github.com/ron-wettenstein/woodelf.git \
    /content/woodelf_explainer

# Confirm woodelf branch
!git -C /content/woodelf_explainer branch --show-current

Cloning into '/content/treebranchmarks'...
remote: Enumerating objects: 623, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 623 (delta 18), reused 27 (delta 12), pack-reused 571 (from 1)
Receiving objects: 100% (623/623), 384.70 KiB | 8.18 MiB/s, done.
Resolving deltas: 100% (413/413), done.
Cloning into '/content/woodelf_explainer'...
remote: Enumerating objects: 1334, done.
remote: Counting objects: 100% (338/338), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 1334 (delta 307), reused 310 (delta 291), pack-reused 996 (from 1)
Receiving objects: 100% (1334/1334), 11.73 MiB | 45.86 MiB/s, done.
Resolving deltas: 100% (993/993), done.
feature/hybrid_background_shap


In [ ]:
# ── Step 4: Install packages ─────────────────────────────────────────────────
import sys

!pip install -q -e /content/woodelf_explainer
!pip install -q -e /content/treebranchmarks

for path in ['/content/woodelf_explainer', '/content/treebranchmarks']:
    if path not in sys.path:
        sys.path.insert(0, path)

# Verify
import woodelf
from woodelf.woodelf_sparse import woodelf_sparse
from woodelf.core.path_to_s_vectors.mn_background_p2s import MNBackgroundShapleyDirectPathToSVectors
print(f'woodelf imported OK (version: {woodelf.__version__})')
print('woodelf_sparse OK')
print('MNBackgroundShapleyDirectPathToSVectors OK')

In [ ]:
# ── Step 5: Restore method caches from Drive (resume after interruption) ──────
import shutil, pathlib

cache_dir = pathlib.Path(f'/content/treebranchmarks/cache/method_results/{EXPERIMENT_NAME}')
cache_dir.mkdir(parents=True, exist_ok=True)

restored = 0
for f in DRIVE_CACHE.glob('*.json'):
    dest = cache_dir / f.name
    if not dest.exists():
        shutil.copy(f, dest)
        print(f'Restored {f.name} ({f.stat().st_size // 1024} KB)')
        restored += 1
    else:
        print(f'Already present: {f.name}')

if not restored:
    print('No cached results to restore — starting fresh')

In [ ]:
# ── Step 6: Run the experiment (cache written to Drive after each approach) ───
%cd /content/treebranchmarks
!python -u -m {EXPERIMENT_MODULE} --result_location {DRIVE_CACHE}

In [ ]:
# ── Step 7: Save HTML report to Drive and download ───────────────────────────
import shutil, pathlib
from google.colab import files

html = pathlib.Path(f'/content/treebranchmarks/results/{EXPERIMENT_NAME}.html')
dest = DRIVE_FOLDER / html.name
shutil.copy(html, dest)
print(f'Saved to Drive: {dest}')
files.download(str(html))